# ViT5 local inference smoke test

Run pretrained ViT5 on a few local records, calculate ROUGE, and save the run artifact. This notebook does not fine-tune the model.

In [1]:
from pathlib import Path
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)
LOGGER = logging.getLogger('vit5-inference')

def is_project(path: Path) -> bool:
    return (path / 'src' / 'data.py').is_file()

cwd = Path.cwd().resolve()
candidates = (cwd, cwd.parent, cwd / 'tuan 3-4')
PROJECT_ROOT = next((path for path in candidates if is_project(path)), None)
if PROJECT_ROOT is None:
    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(f'Project root not found. Checked:\n{checked}')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
LOGGER.info('project_root=%s', PROJECT_ROOT)

21:13:23 | INFO | project_root=/home/dungcony/projects/python/thuc-tap/tuan 3-4


In [2]:
# Run once for the current local virtual environment.
%pip install -e "{PROJECT_ROOT}"

Obtaining file:///home/dungcony/projects/python/thuc-tap/tuan%203-4
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vn-summarization-week34 (pyproject.toml) ... done
  Created wheel for vn-summarization-week34: filename=vn_summarization_week34-0.1.0-0.editable-py3-none-any.whl size=3798 sha256=b65d6541370042d7bb27990f174a02a41b0e42aad5a41cd0bfb0c1b5fc32788c
  Stored in directory: /tmp/pip-ephem-wheel-cache-vonl9ycu/wheels/85/5f/93/0fd90ad158d1f25ccf35560e28113b20e5c4f014ea8d1f42d6
Successfully built vn-summarization-week34
  Attempting uninstall: vn-summarization-week34
    Found existing installation: vn-summarization-week34 0.1.0
    Not uninstalling vn-summarization-week34 at /home/dungcony/projects/python/thuc-tap/tuan 3-4/src, outside environment /home/dungcony/projects/python/thuc-tap/.venv
    Can'

In [3]:
import torch

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'vietnews_medical_raw_1000.csv'
OUTPUT_PATH = PROJECT_ROOT / 'results' / 'local_inference_5.json'
MODEL_NAME = 'VietAI/vit5-base-vietnews-summarization'
MAX_SAMPLES = 5

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LOGGER.info('device=%s', DEVICE)
if DEVICE == 'cuda':
    LOGGER.info('gpu=%s', torch.cuda.get_device_name(0))

21:13:28 | INFO | device=cuda
21:13:28 | INFO | gpu=NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
from data import read_rows, records_from_rows

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

records, audit = records_from_rows(
    read_rows(DATA_PATH),
    dataset_name='vietnews-medical',
    keep_at_most=MAX_SAMPLES,
)
articles = [record.article for record in records]
references = [record.summary for record in records]
LOGGER.info('input_rows=%d accepted_samples=%d', audit['input_rows'], len(records))

21:13:28 | INFO | input_rows=990 accepted_samples=5


In [5]:
from model import generate_summaries
from metrics import compression_statistics, compute_rouge

predictions, used_device = generate_summaries(
    articles,
    model_name=MODEL_NAME,
    device=DEVICE,
    prefix='summarize: ',
    batch_size=1,
    max_source_length=384,
    max_new_tokens=64,
    min_new_tokens=8,
    num_beams=2,
    no_repeat_ngram_size=3,
)
rouge = compute_rouge(predictions, references)
compression = compression_statistics(articles, predictions)

LOGGER.info('inference complete samples=%d device=%s', len(predictions), used_device)
LOGGER.info('rouge1_f1=%.2f rouge2_f1=%.2f rougeL_f1=%.2f', rouge['rouge1']['f1'], rouge['rouge2']['f1'], rouge['rougeL']['f1'])

/home/dungcony/projects/python/thuc-tap/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
21:13:38 | INFO | Using default tokenizer.
21:13:38 | INFO | inference complete samples=5 device=cuda
21:13:38 | INFO | rouge1_f1=66.01 rouge2_f1=36.11 rougeL_f1=41.58


In [6]:
import json
from datetime import datetime, timezone

payload = {
    'run': {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model': MODEL_NAME,
        'device': used_device,
        'number_of_examples': len(records),
    },
    'generation': {
        'prefix': 'summarize: ',
        'max_source_length': 384,
        'max_new_tokens': 64,
        'min_new_tokens': 8,
        'num_beams': 2,
        'no_repeat_ngram_size': 3,
    },
    'metrics': {'rouge': rouge, 'compression': compression},
    'predictions': [
        {
            'id': record.id,
            'reference': record.summary,
            'prediction': prediction,
        }
        for record, prediction in zip(records, predictions)
    ],
}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
LOGGER.info('result_written=%s', OUTPUT_PATH)

21:13:38 | INFO | result_written=/home/dungcony/projects/python/thuc-tap/tuan 3-4/results/local_inference_5.json
